# Train CIFAR Lens on a free Colab GPU

Select T4 GPU and runtime version 2026.01 (Python 3.12) for the tested TensorFlow 2.19 / Keras 3.10 environment.
Checkpoints save locally by default. Optionally mount Google Drive yourself before running to persist them across runtime deletion.
Repeat seeds 73, 1 and 2 for a multi-seed comparison. The 91% accuracy goal is unmeasured until final evaluation completes.


In [ ]:
# Optional: run these lines yourself to persist checkpoints in Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
import sys, subprocess, os
from pathlib import Path
assert sys.version_info[:2] == (3, 12), 'Select runtime version 2026.01 (Python 3.12)'
repo = Path('/content/cifar-lens-corrected')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/amirhamzaaaa4/cifar-lens.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'tensorflow==2.19.0', 'keras==3.10.0', 'numpy==2.1.3', 'scikit-learn>=1.6,<2', 'Pillow>=11,<13'], check=True)
import tensorflow as tf
assert tf.config.list_physical_devices('GPU'), 'Select T4 GPU'
print('TensorFlow', tf.__version__, 'GPU:', tf.config.list_physical_devices('GPU'))
tf.keras.utils.get_file('cifar-10-batches-py-target_archive', origin='https://zenodo.org/records/10089977/files/cifar-10-python.tar.gz?download=1', file_hash='6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce', hash_algorithm='sha256', cache_subdir='datasets')


## Train the corrected ResNet20 candidate

In [ ]:
import subprocess, sys
SEED = 73
EPOCHS = 180
base = Path('/content/drive/MyDrive/cifar-lens') if Path('/content/drive/MyDrive').exists() else Path('artifacts')
out = base / f'resnet20-corrected-s{SEED}'
if (out / 'results.json').exists():
    print('Already finished:', out)
else:
    command = [sys.executable, '-u', '-m', 'lens.train', '--architecture', 'resnet20', '--epochs', str(EPOCHS), '--label-smoothing', '0.1', '--cutout', '8', '--tta', '--mixed-precision', '--seed', str(SEED), '--output', str(out)]
    if (out / 'latest.keras').exists():
        command.append('--resume')
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('Training failed; inspect the output above')


## Optional: reference models under the same split

In [ ]:
RUN_REFERENCES = False  # Set True to train the optional baseline and residual models.
if RUN_REFERENCES:
    for arch in ('baseline', 'residual'):
        ref = base / arch
        if not (ref / 'results.json').exists():
            command = [sys.executable, '-m', 'lens.train', '--architecture', arch, '--epochs', '50', '--output', str(ref)]
            if (ref / 'latest.keras').exists():
                command.append('--resume')
            subprocess.run(command, check=True)


## Compare everything that has finished

In [ ]:
done = [str(p.parent) for p in base.glob('*/results.json')]
if done:
    subprocess.run([sys.executable, '-m', 'lens.compare', *done, '--output', str(base / 'benchmark')], check=True)
    from IPython.display import Image, display
    display(Image(str(base / 'benchmark/accuracy.png')))
else:
    print('No completed experiments yet')


## Package the model for a GitHub Release

Creates a zip without the training checkpoints. Upload it as a release asset, then deploy with `LENS_EXPERIMENT_URL=<asset url>` or install locally with `python -m lens.fetch <asset url> --output artifacts/resnet20`.

In [ ]:
import shutil, tempfile
keep = ['model.keras', 'results.json', 'errors.npz', 'epochs.csv', 'samples']
with tempfile.TemporaryDirectory() as tmp:
    target = Path(tmp) / out.name
    target.mkdir()
    for name in keep:
        src = out / name
        if src.is_dir():
            shutil.copytree(src, target / name)
        elif src.exists():
            shutil.copy2(src, target / name)
    archive = shutil.make_archive(f'/content/{out.name}', 'zip', tmp)
print('Release asset:', archive)
from google.colab import files
files.download(archive)
